# Campaign Planning Agents - v1.4
Run these cells in order. Begin with practice mode. AI mode downloads models and runs inference on your Colab runtime; no paid API key is used. Free runtime resources are not guaranteed.

If Colab reports a conflict with its preinstalled diffusers library, our project does not use diffusers. In a session dedicated to this project, run `%pip uninstall -y diffusers`, restart the session, then rerun setup. Do not upgrade huggingface-hub independently of Transformers.


In [ ]:
%pip install "torch>=2.2,<3" "transformers>=4.45,<5" "sentence-transformers>=3,<6" "gradio>=5,<7" "httpx[socks]>=0.28,<1"

## Upload the project ZIP
Select Campaign_Planning_Agents_v1_4_2026-10-11.zip. This extracts only the known project folder.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import sys

uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith(".zip"))
base = Path("/content").resolve()
with zipfile.ZipFile(zip_name) as archive:
    for item in archive.infolist():
        target = (base / item.filename).resolve()
        if not target.is_relative_to(base / "campaign-planning-agent"):
            raise ValueError("Unexpected archive path")
    archive.extractall(base)
project = base / "campaign-planning-agent"
sys.path.insert(0, str(project))
import os
os.chdir(project)
print("Project folder:", Path.cwd())
assert (project / "workflow.py").exists()
assert list((project / "data").glob("*.md"))
assert (project / "VERSION.txt").read_text().strip() == "1.4", "Upload the v1.4 project ZIP."


## Run project tests before generation

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=project, check=True)

## Practice mode
These outputs are scripted, not AI-generated. Read the handoffs and inspect the files.

In [ ]:
from workflow import Brief, run_campaign

brief = Brief(product="TrailSip reusable water bottle", audience="College students",
              goal="Increase product-page visits and purchases", channel="Instagram",
              budget=200, days=14)
state, output_files = run_campaign(brief, mode="practice")
print("\n".join(state.trace))
print(Path(output_files[0]).read_text())

## Optional: AI mode
First use downloads models. CPU inference can be slow. Outputs need factual review. This cell is the actual LLM and embedding-based RAG workflow.
AI mode makes five short calls. Accepted selections and labeled Python fallbacks are recorded in workflow_trace.json. The report uses approved quotes and template copy, not unrestricted AI writing.


In [ ]:
from workflow import LocalModel, Retriever
ai_model = LocalModel()
ai_retriever = Retriever(semantic=True)
state, output_files = run_campaign(brief, mode="ai", model=ai_model, retriever=ai_retriever, progress=print)
print("\n".join(state.trace))
print(Path(output_files[0]).read_text())

## Download the latest run
Run after practice mode or AI mode to save your report, calendar and trace.

In [ ]:
import shutil
run_folder = Path(output_files[0]).parent
archive_path = shutil.make_archive("/content/my_campaign_outputs", "zip", run_folder)
files.download(archive_path)

## Optional browser demo
This creates a temporary public share link. Keep the runtime running. Stop the cell to close the demo.

In [ ]:
from app import demo, resources
if "ai_model" in globals() and "ai_retriever" in globals():
    resources.update(model=ai_model, retriever=ai_retriever)
demo.queue().launch(share=True, debug=True)